# M6.2 Mutual Inductance by the Neumann Formula

Book pointer: chapter 15, sections 15.1 (inductance), 15.2 (mutual inductance), 15.4 (energy in magnetic fields), 15.5 (magnetic forces).

## 1. Recall

Answer on paper before running any cell.

1. Define the mutual inductance $M_{21}$ of two loops through flux linkage. What are its units, and why does it depend only on geometry?
2. Write the Neumann formula for $M$ as a double line integral. Which property of the formula makes $M_{12} = M_{21}$ obvious?
3. Two small coaxial loops of radii $a$ and $b$ a distance $d$ apart, $d \gg a, b$: what is $M$, and how does it fall with $d$?
4. Write the magnetic energy of two loops carrying $I_1$ and $I_2$ in terms of $L_1$, $L_2$ and $M$.
5. How do you get the force between two loops from $M$? Do parallel currents in coaxial loops attract or repel?

## 2. Principle to algorithm

A current $I_1$ in loop 1 makes a field $\mathbf{B}_1 \propto I_1$. Its flux through loop 2 is therefore proportional to $I_1$ too, and the constant is the mutual inductance, $\Phi_{21} = M_{21}I_1$. There are three ways to compute it, and this notebook uses all three so that each checks the others.

1. **Flux route.** Compute $\mathbf{B}_1$ with the Biot-Savart sum (`bs.wire_B`) and integrate $\mathbf{B}_1\cdot\hat{\mathbf{n}}$ over a flat disk spanning loop 2 (`ind.disk_flux`). Cost: one Biot-Savart sum per quadrature point, a few thousand points. Error source: if loop 1 passes close to loop 2's rim, the integrand is nearly singular near the rim and the polar quadrature needs many points there.
2. **Vector-potential route.** Since $\mathbf{B} = \nabla\times\mathbf{A}$, Stokes' theorem turns the flux into a line integral, $\Phi_{21} = \oint_2\mathbf{A}_1\cdot d\mathbf{l}_2$. Evaluate $\mathbf{A}_1$ with `bs.wire_A` at the midpoints of loop 2's segments and sum $\mathbf{A}_1\cdot\Delta\mathbf{l}_2$. Cost: one sum per segment of loop 2.
3. **Neumann route.** Insert $\mathbf{A}_1 = \dfrac{\mu_0I_1}{4\pi}\oint_1\dfrac{d\mathbf{l}_1}{|\mathbf{r}_2 - \mathbf{r}_1|}$ into route 2:
$$M = \frac{\mu_0}{4\pi}\oint_1\oint_2\frac{d\mathbf{l}_1\cdot d\mathbf{l}_2}{|\mathbf{r}_1 - \mathbf{r}_2|}.$$
The formula is symmetric in the two loops, so $M_{12} = M_{21} = M$. On polylines it is a double sum over segment pairs (`ind.neumann_M`), the same arithmetic as route 2 written once. Error source: a segment pair closer than about one segment length is badly represented by its midpoints, so the loops must stay farther apart than a segment.

For coaxial circular loops there is an exact answer in elliptic integrals (`ind.coaxial_loops_M`), which is the test case. Far apart, each loop looks like a dipole to the other and $M \to \mu_0\pi a^2b^2/(2d^3)$.

Energy and force. Two loops carrying $I_1$ and $I_2$ store $U = \tfrac12L_1I_1^2 + \tfrac12L_2I_2^2 + MI_1I_2$. If loop 2 moves by $\delta d$ while both currents are held fixed by their sources, the sources supply $2\,\delta U$, half goes into the field and half into mechanical work, so the force is $F = +I_1I_2\,\partial M/\partial d$ (with the plus sign; the minus sign belongs to the case of fixed fluxes). For coaxial loops with parallel currents $M$ falls with $d$, so the force on loop 2 points back towards loop 1: parallel currents attract. The direct check is the Lorentz force $\sum I_2\,\Delta\mathbf{l}_2\times\mathbf{B}_1$ on loop 2.

### Animation

Two coaxial loops. The current in loop 1 rises and falls slowly; its field lines thread loop 2, and the marker on loop 2 shows the induced current opposing each change. The readout gives $M$ and the emf $-M\,dI_1/dt$. Then loop 2 moves away and $M$ drops, traced on a small plot of $M$ against separation, which is the curve computed in section 3.1.

In [ ]:
from emlib import show
show.animation("m6_waves", "CoupledCoils")

## 3. Build

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from emlib import biot_savart as bs
from emlib import induction as ind
from emlib.biot_savart import MU0

plt.rcParams["figure.figsize"] = (6, 4.5)
R = 0.1          # radius of loop 1 [m]
N_SEG = 400      # segments per loop for the large loops

### 3.1 Coaxial loops: Neumann against the exact result

Two equal loops of radius $R = 10\,$cm, coaxial, separated by $d$ from $0.05R$ to $10R$. The Neumann double sum with 400 segments per loop is compared with the elliptic-integral formula and with the dipole limit.

In [ ]:
d_over_R = np.logspace(np.log10(0.05), 1, 25)
M_num = np.array([ind.neumann_M(bs.loop(R, N_SEG), bs.loop(R, N_SEG, centre=(0, 0, d * R))) for d in d_over_R])
M_exact = ind.coaxial_loops_M(R, R, d_over_R * R)
M_dipole = MU0 * np.pi * R**4 / (2 * (d_over_R * R) ** 3)
rel = M_num / M_exact - 1

print("  d/R      M Neumann [H]   M exact [H]     rel. error")
for k in (0, 6, 12, 18, 24):
    print(f"{d_over_R[k]:6.2f}   {M_num[k]:.6e}   {M_exact[k]:.6e}   {rel[k]:+.1e}")
print(f"largest |relative error| over the sweep: {np.abs(rel).max():.1e}")
print(f"polygon area deficit per loop, 1 - sin(2 pi/N)/(2 pi/N): {1 - np.sin(2 * np.pi / N_SEG) / (2 * np.pi / N_SEG):.2e}")
for dd in (3.0, 5.0, 10.0):
    print(f"dipole limit / exact at d = {dd:4.1f} R: {MU0 * np.pi * R**4 / (2 * (dd * R) ** 3) / ind.coaxial_loops_M(R, R, dd * R):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].loglog(d_over_R, M_exact, "k-", label="exact (elliptic integrals)")
ax[0].loglog(d_over_R, M_num, "C1o", ms=4, label="Neumann sum, 400 segments")
ax[0].loglog(d_over_R, M_dipole, "C0--", label="dipole limit  mu0 pi R^4 / 2 d^3")
ax[0].set_xlabel("d / R"); ax[0].set_ylabel("M [H]"); ax[0].legend(); ax[0].set_title("coaxial loops of equal radius")
ax[1].semilogx(d_over_R, rel, "C1o-", ms=4)
ax[1].axhline(-2 * (1 - np.sin(2 * np.pi / N_SEG) / (2 * np.pi / N_SEG)), color="grey", ls=":", label="two polygon area deficits")
ax[1].set_xlabel("d / R"); ax[1].set_ylabel("relative error"); ax[1].legend(); ax[1].set_title("error of the double sum")
plt.tight_layout(); plt.show()

The error is largest far away, not close up, and it settles at $-8\times10^{-5}$. That number is the polygon effect: a 400-sided inscribed polygon has an area smaller than the circle by $1 - \sin(2\pi/N)/(2\pi/N) = 4.1\times10^{-5}$, and in the dipole limit $M$ is proportional to the product of the two areas, so the deficit counts twice. Close up, $M$ is set by the nearby parts of the two wires, where both polygons sit at the same radius and the deficit largely cancels.

The dipole limit is approached slowly: it is 33 percent too high at $d = 3R$, 12 percent at $5R$ and 3 percent at $10R$. The leading correction is of order $(R/d)^2$ with a coefficient of about 3, because each loop has the same size as the other and neither sees the other's field as uniform.

### 3.2 A small loop moved sideways: $M$ changes sign

A small loop of radius $b = 2\,$cm, parallel to the large loop and $2\,$cm above its plane, slides from the axis out to $2.5R$. Inside the large loop its field points up through the small loop; outside, the field lines of the large loop are on their way back down, because every field line closes around the wire. So the flux through the small loop, and with it $M$, must change sign somewhere near the wire. The line in the plot is $B_z$ of the large loop at the small loop's centre times the small loop's area, the small-loop limit of $M$.

In [ ]:
b, h = 0.02, 0.02
offsets = np.linspace(0, 2.5 * R, 41)
M_off = np.array([ind.neumann_M(bs.loop(R, N_SEG), bs.loop(b, 120, centre=(s, 0, h))) for s in offsets])
m1, d1 = bs.segments(bs.loop(R, N_SEG))
Bz_centre = bs.wire_B(m1, d1, 1.0, np.stack([offsets, 0 * offsets, h + 0 * offsets], axis=-1))[:, 2]

k = np.nonzero(np.diff(np.sign(M_off)))[0][0]
s0 = offsets[k] - M_off[k] * (offsets[k + 1] - offsets[k]) / (M_off[k + 1] - M_off[k])
print(f"M on the axis: {M_off[0]:.3e} H;  M at 2R: {M_off[np.argmin(np.abs(offsets - 2 * R))]:.3e} H")
print(f"M changes sign at offset {s0 / R:.3f} R (the small loop's centre is then just outside the wire)")

plt.plot(offsets / R, M_off * 1e9, "C1o", ms=4, label="Neumann sum")
plt.plot(offsets / R, Bz_centre * np.pi * b**2 * 1e9, "k-", lw=1, label="B_z(centre) x area")
plt.axhline(0, color="grey", lw=0.8); plt.axvline(1, color="grey", ls=":", label="radius of the large loop")
plt.xlabel("offset / R"); plt.ylabel("M [nH]"); plt.legend(); plt.title("small loop moved across the large loop's wire")
plt.show()

Away from the wire the two curves agree to about 2 percent, the size of the field's variation across a loop of radius $b = 0.2R$. Near the wire the field changes sharply across the small loop and the centre value is no longer a good average: at offset $1.1R$ the centre estimate is seven times the true $M$. The zero crossing sits a little outside $R$: just beyond the wire the downward return field near the wire wins over the upward field of the far side of the loop.

### 3.3 Tilt: $M \propto \cos\theta$ in the dipole limit

The small loop now sits on the axis, $5\,$cm above the centre of the large loop, and is tilted by $\theta$ about the $y$ axis. If the field is uniform over the small loop, its flux is $\mathbf{B}\cdot\hat{\mathbf{n}}\,\pi b^2$ and $M(\theta) = M(0)\cos\theta$. A loop of radius $1\,$cm is small enough for this to hold to a few parts in $10^5$.

In [ ]:
thetas = np.radians([0, 15, 30, 45, 60, 75, 90])
M_tilt = np.array([ind.neumann_M(bs.loop(R, N_SEG), bs.loop(0.01, 120, centre=(0, 0, 0.05), normal=(np.sin(t), 0, np.cos(t)))) for t in thetas])
print(" theta    M / M(0)     cos(theta)")
for t, m in zip(thetas, M_tilt / M_tilt[0]):
    print(f"{np.degrees(t):5.0f}   {m:+.6f}   {np.cos(t):+.6f}")

### 3.4 Reciprocity and the three routes

A non-coaxial, unequal pair: loop 1 of radius $10\,$cm in the $xy$ plane, loop 2 of radius $5\,$cm centred at $(4, 2, 6)\,$cm and tilted. Reciprocity says the flux of loop 1's field through loop 2 per unit current equals the flux of loop 2's field through loop 1 per unit current, even though the two fields and the two surfaces look nothing alike. The Neumann formula makes this a symmetry of the sum. The table compares all three routes.

In [ ]:
c2 = np.array([0.04, 0.02, 0.06])
n2 = np.array([0.3, 0.0, 1.0]); n2 /= np.linalg.norm(n2)
b2 = 0.05
loop1, loop2 = bs.loop(R, 300), bs.loop(b2, 300, centre=c2, normal=n2)
m1, d1 = bs.segments(loop1); m2, d2 = bs.segments(loop2)

routes = {}
t0 = time.perf_counter(); routes["Neumann, M12"] = ind.neumann_M(loop1, loop2); t_n = time.perf_counter() - t0
routes["Neumann, M21"] = ind.neumann_M(loop2, loop1)
t0 = time.perf_counter(); routes["flux of B1 through loop 2"] = ind.disk_flux(lambda p: bs.wire_B(m1, d1, 1.0, p), c2, n2, b2); t_f = time.perf_counter() - t0
t0 = time.perf_counter(); routes["circulation of A1 around loop 2"] = np.sum(bs.wire_A(m1, d1, 1.0, m2) * d2); t_a = time.perf_counter() - t0
routes["flux of B2 through loop 1"] = ind.disk_flux(lambda p: bs.wire_B(m2, d2, 1.0, p), (0, 0, 0), (0, 0, 1), R)
timing = {"Neumann, M12": t_n, "flux of B1 through loop 2": t_f, "circulation of A1 around loop 2": t_a}

ref = routes["Neumann, M12"]
print("route                              M [H]          relative to Neumann   time")
for name, val in routes.items():
    tt = f"{timing[name] * 1e3:7.1f} ms" if name in timing else ""
    print(f"{name:33s}  {val:.8e}   {val / ref - 1:+.1e}            {tt}")

The Neumann sums in the two orders agree to rounding, and the circulation of $\mathbf{A}$ is the same arithmetic. The two flux routes differ at the $10^{-4}$ level, set by the quadrature over the disk and by the field being evaluated near a polygon rather than a circle. The flux route is also about a hundred times slower, because it evaluates a three-component field at thousands of interior points instead of a scalar sum over segment pairs. That is why the Neumann formula is the practical tool, even though the flux definition is the one with the clearest meaning.

### 3.5 The force between two coaxial loops

Two equal coaxial loops with $I_1 = I_2 = 1\,$A. The force on loop 2 along the axis is computed three ways: $I_1I_2\,\partial M/\partial d$ from the exact formula by a central difference, the same from the Neumann sum, and directly as $\sum I_2\,\Delta\mathbf{l}_2\times\mathbf{B}_1$ over loop 2's segments.

In [ ]:
def force_from_M(Mfunc, d, I1=1.0, I2=1.0, rel_step=1e-4):
    h = rel_step * R
    return I1 * I2 * (Mfunc(d + h) - Mfunc(d - h)) / (2 * h)

def force_direct(d, I1=1.0, I2=1.0, n=N_SEG):
    m1, d1 = bs.segments(bs.loop(R, n)); m2, d2 = bs.segments(bs.loop(R, n, centre=(0, 0, d)))
    return np.sum(I2 * np.cross(d2, bs.wire_B(m1, d1, I1, m2)), axis=0)

seps = np.array([0.1, 0.3, 1.0, 3.0]) * R
print("  d/R    F from exact dM/dd [N]   F from Neumann dM/dd [N]   F direct (x, y, z) [N]")
F_exact, F_dir = [], []
for d in seps:
    fe = force_from_M(lambda s: ind.coaxial_loops_M(R, R, s), d)
    fn = force_from_M(lambda s: ind.neumann_M(bs.loop(R, N_SEG), bs.loop(R, N_SEG, centre=(0, 0, s))), d)
    fd = force_direct(d)
    F_exact.append(fe); F_dir.append(fd[2])
    print(f"{d / R:5.1f}   {fe:+.6e}            {fn:+.6e}            ({fd[0]:+.0e}, {fd[1]:+.0e}, {fd[2]:+.6e})")
F_exact, F_dir = np.array(F_exact), np.array(F_dir)
print("negative F_z: the force on loop 2 points back towards loop 1, so parallel currents attract")

### Exercise: mutual inductance by the flux route

Replace the body of `M_by_flux` with your own version. It takes the vertices of loop 1 and the centre, unit normal and radius of a circular loop 2, and returns the flux of loop 1's field (for $1\,$A) through the flat disk spanning loop 2. The check compares it with the Neumann sum for a non-coaxial pair.

In [ ]:
def M_by_flux(loop1, loop2_centre, loop2_normal, loop2_R):
    # replace from here with your own version
    mid, dl = bs.segments(loop1)
    return ind.disk_flux(lambda p: bs.wire_B(mid, dl, 1.0, p), loop2_centre, loop2_normal, loop2_R)


c_ex = np.array([-0.03, 0.05, 0.08]); n_ex = np.array([0.0, -0.4, 1.0]); n_ex /= np.linalg.norm(n_ex)
M_flux = M_by_flux(bs.loop(R, 300), c_ex, n_ex, 0.04)
M_neu = ind.neumann_M(bs.loop(R, 300), bs.loop(0.04, 300, centre=c_ex, normal=n_ex))
print(f"flux route {M_flux:.8e} H    Neumann {M_neu:.8e} H    relative difference {M_flux / M_neu - 1:+.1e}")
assert abs(M_flux / M_neu - 1) < 1e-3

## 4. Verify

Four closed-form or structural checks:

1. the Neumann sum against the elliptic-integral formula at five separations;
2. reciprocity for the unequal, non-coaxial pair of section 3.4;
3. the force from $\partial M/\partial d$ against the direct Lorentz-force sum;
4. the sign change of $M$ for the small loop moved across the large loop's wire.

In [ ]:
check_d = np.array([0.05, 0.2, 1.0, 3.0, 10.0]) * R
errs = [ind.neumann_M(bs.loop(R, N_SEG), bs.loop(R, N_SEG, centre=(0, 0, d))) / ind.coaxial_loops_M(R, R, d) - 1 for d in check_d]
print("Neumann vs exact at d/R =", check_d / R, ":", np.array2string(np.array(errs), precision=1))
assert np.all(np.abs(errs) < 1e-3)

recip = routes["Neumann, M21"] / routes["Neumann, M12"] - 1
print(f"reciprocity M21/M12 - 1 = {recip:.1e}")
assert abs(recip) < 1e-12

force_err = F_dir / F_exact - 1
print("direct force / force from dM/dd - 1:", np.array2string(force_err, precision=1))
assert np.all(np.abs(force_err) < 1e-2)

print(f"M at offset 0.5R: {M_off[np.argmin(np.abs(offsets - 0.5 * R))]:+.3e} H,  at 1.5R: {M_off[np.argmin(np.abs(offsets - 1.5 * R))]:+.3e} H")
assert M_off[np.argmin(np.abs(offsets - 0.5 * R))] > 0 and M_off[np.argmin(np.abs(offsets - 1.5 * R))] < 0

plt.loglog(seps / R, -F_exact, "k-", label="I1 I2 dM/dd (exact M)")
plt.loglog(seps / R, -F_dir, "C1o", label="direct sum of I dl x B")
plt.xlabel("d / R"); plt.ylabel("attractive force [N]"); plt.legend(); plt.title("force between coaxial loops, 1 A each")
plt.show()

## 5. Explore and rebuild

Explore. The self-inductance of a single ring cannot come from the Neumann formula directly: with both line integrals on the same curve the integrand diverges where $\mathbf{r}_1 = \mathbf{r}_2$. The classic way round is to give the wire its real radius $a$ and evaluate the formula between the wire's centreline (a loop of radius $R$) and a parallel path on its surface (a coplanar loop of radius $R - a$). Do this for $R = 10\,$cm and $a = 1\,$mm, with enough segments that a segment is shorter than $a$, and compare with `ind.ring_self_L(R, a)` $= \mu_0R(\ln(8R/a) - 7/4)$ and with $\mu_0R(\ln(8R/a) - 2)$. Which of the two does the offset-path result reproduce, and what does the difference $\mu_0R/4$ represent? (Hint: the offset path counts only the flux outside the wire.) How does the answer change if the current crowds to the surface of the wire at high frequency?

Rebuild. Close this notebook. From a blank editor, write the Neumann double sum for two polylines (midpoints and steps, a pairwise distance array, one matrix product) and the flux route (a polar grid on a disk spanning loop 2, the Biot-Savart field at every grid point, the normal component weighted by $r\,\Delta r\,\Delta\phi$). Reproduce the table of section 3.4.